In [1]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [2]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql", "sql/01_clean.sql", "sql/02_features.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

feat = load_queries("sql/02_features.sql")
display(run(feat, "features_reconcile").T)

,0
n_rows,20405666
pu_zone_null,0
pu_borough_null,0
min_hour,0
max_hour,23
min_isodow,1
max_isodow,7
n_pickup_zones,262
min_request_date,2024-12-31 00:00:00
max_request_date,2025-02-01 00:00:00


In [3]:
import pandas as pd
p = load_queries("sql/02f_driver_pay.sql")
con.execute(p["create_trips_pay"])
rec = run(p, "pay_reconcile"); display(rec.T)

,0,1,2
company,Uber,Lyft,All
n_rows,15356455,5049211,20405666
n_five_borough,15355926,5048966,20404892
n_pay_null,0,0,0
n_pay_negative,313,0,313
n_pay_zero,4225,0,4225
n_pay_positive,15351917,5049211,20401128
n_zero_miles,2398,208,2606
n_zero_time,4,0,4
n_in_pay_per_mile,15354057,5049003,20403060


In [4]:
pay_dist = run(p, "pay_distribution_by_company"); display(pay_dist)
pay_mean = run(p, "pay_mean_sensitivity"); display(pay_mean)

,company,n,mean_pay,min_pay,p01,p05,p10,p25,p50,p75,p90,p95,p99,max_pay
0,Uber,15356455,18.937,-55.02,4.00,4.66,5.67,8.43,14.16,23.97,37.66,48.99,77.98,994.20
1,Lyft,5049211,17.781,4.00,5.47,5.47,5.47,8.13,13.32,22.03,34.56,45.91,74.02,525.53


,company,n,mean_all,median,mean_trimmed_1_99,mean_positive_pay_only,max_pay,pct_of_total_pay_from_top_1pct_trips,pct_of_trips_in_top_1pct
0,Uber,15356455,18.937,14.16,18.119,18.943,994.20,5.63,1.0
1,Lyft,5049211,17.781,13.32,17.070,17.781,525.53,5.56,1.0


In [5]:
ppm = run(p, "ppm_distribution_by_company"); display(ppm)
ppmin = run(p, "ppmin_distribution_by_company"); display(ppmin)
diag = run(p, "ratio_denominator_diagnostics"); display(diag)

,company,n_rows,n_eligible,n_no_ratio_zero_miles,mean_ppm,p01,p05,p10,p25,p50,p75,p90,p95,p99,max_ppm
0,Uber,15356455,15354057,2398,5.234,2.209,2.576,2.817,3.50,4.596,5.893,7.836,9.787,16.412,7110.0
1,Lyft,5049211,5049003,208,5.269,2.406,2.715,2.984,3.71,4.731,5.978,7.717,9.390,14.985,5470.0


,company,n_rows,n_eligible,n_no_ratio_zero_time,mean_ppmin,p01,p05,p10,p25,p50,p75,p90,p95,p99,max_ppmin
0,Uber,15356455,15356451,4,1.025,0.527,0.724,0.758,0.819,0.929,1.127,1.385,1.612,2.365,270.6
1,Lyft,5049211,5049211,0,0.987,0.689,0.730,0.755,0.806,0.893,1.059,1.290,1.478,2.065,1378.2


,metric,company,denominator_bucket,n,ratio_p50,ratio_p99,ratio_max
0,pay_per_mile,Uber,1_<0.1 mi,3098,108.000,2101.070,7110.00
1,pay_per_mile,Uber,2_0.1-<0.25 mi,12824,23.529,137.348,535.95
2,pay_per_mile,Uber,3_0.25-<0.5 mi,180156,11.735,46.250,361.39
3,pay_per_mile,Uber,4_0.5-<1 mi,1556487,6.963,24.736,272.49
4,pay_per_mile,Uber,5_1+ mi,13601492,4.365,12.179,82.79
5,pay_per_mile,Lyft,1_<0.1 mi,292,195.357,5256.800,5470.00
6,pay_per_mile,Lyft,2_0.1-<0.25 mi,4058,26.946,124.993,220.95
7,pay_per_mile,Lyft,3_0.25-<0.5 mi,65981,13.181,43.280,225.57
8,pay_per_mile,Lyft,4_0.5-<1 mi,548526,7.453,20.945,99.96
9,pay_per_mile,Lyft,5_1+ mi,4430146,4.490,10.609,83.86


In [6]:
dist = run(p, "pay_by_distance_band"); display(dist)
dur = run(p, "pay_by_duration_band"); display(dur)

,company,distance_band,n,pct_of_company,pay_p25,pay_p50,pay_p75,ppm_p50,ppmin_p50,median_miles,median_minutes
0,Uber,0_zero miles,2398,0.02,2.00,4.34,7.04,NaN,1.077,0.00,3.63
1,Uber,1_<1 mi,1752565,11.41,4.47,5.13,6.68,7.295,0.956,0.77,5.42
2,Uber,2_1-<2 mi,3852738,25.09,6.59,8.09,10.11,5.529,0.836,1.44,9.27
3,Uber,3_2-<3 mi,2284001,14.87,10.28,12.11,14.51,4.945,0.859,2.44,13.82
4,Uber,4_3-<5 mi,2614899,17.03,14.06,16.66,19.93,4.342,0.904,3.83,18.30
5,Uber,5_5-<10 mi,2849596,18.56,20.61,24.50,29.14,3.448,1.026,6.91,23.78
6,Uber,6_10-<20 mi,1637745,10.66,33.70,40.33,48.35,2.925,1.180,13.27,34.00
7,Uber,7_20+ mi,362513,2.36,59.43,69.92,88.18,2.625,1.462,25.14,49.40
8,Lyft,0_zero miles,208,0.00,5.47,6.79,9.92,NaN,0.714,0.00,10.53
9,Lyft,1_<1 mi,618857,12.26,5.47,5.47,6.19,7.770,0.971,0.77,5.78


,company,duration_band,n,pct_of_company,pay_p25,pay_p50,pay_p75,ppm_p50,ppmin_p50,median_miles,median_minutes
0,Uber,0_zero time,4,0.00,3.00,4.24,4.55,NaN,NaN,0.00,0.00
1,Uber,1_<5 min,839884,5.47,4.16,4.49,5.28,6.914,1.202,0.69,4.03
2,Uber,2_5-<10 min,3486394,22.70,5.70,6.85,8.15,5.253,0.871,1.25,7.65
3,Uber,3_10-<15 min,3295458,21.46,9.53,11.03,13.13,4.927,0.878,2.18,12.35
4,Uber,4_15-<20 min,2489539,16.21,13.90,16.01,19.23,4.425,0.916,3.53,17.30
5,Uber,5_20-<30 min,2956281,19.25,19.77,23.55,28.46,3.927,0.966,5.88,24.05
6,Uber,6_30-<45 min,1650290,10.75,30.25,36.36,44.89,3.443,1.015,10.42,35.27
7,Uber,7_45+ min,638605,4.16,47.71,58.13,73.63,3.312,1.050,17.70,53.18
8,Lyft,1_<5 min,242759,4.81,5.47,5.47,5.47,8.250,1.356,0.67,4.08
9,Lyft,2_5-<10 min,1129139,22.36,5.47,6.49,7.54,5.343,0.859,1.20,7.70


In [7]:
wb = run(p, "pay_by_wait_band"); display(wb)
wwd = run(p, "pay_by_wait_within_distance")
for val, title in [("ppm_p50", "Median pay per mile"), ("pay_p50", "Median pay per trip"), ("n", "Cell sizes")]:
    print(title, "(rows: company, distance band; columns: wait band)")
    display(wwd.pivot_table(index=["company", "distance_band"], columns="wait_band", values=val))

,company,wait_band,n,pct_of_company_valid,pay_p25,pay_p50,pay_p75,ppm_p50,ppmin_p50,median_miles,median_minutes
0,Uber,1_<2 min,1755078,11.52,7.66,12.49,20.69,4.762,0.904,2.40,13.67
1,Uber,2_2-<4 min,6431054,42.21,8.00,13.17,21.85,4.682,0.909,2.62,14.32
2,Uber,3_4-<6 min,3883360,25.49,8.49,14.25,23.95,4.582,0.932,2.92,15.08
3,Uber,4_6-<8 min,1617045,10.61,9.33,15.95,27.13,4.482,0.969,3.33,16.33
4,Uber,5_8-<10 min,733839,4.82,10.24,17.82,30.49,4.274,1.001,3.94,17.92
5,Uber,6_10-<15 min,624427,4.10,11.03,19.58,33.51,3.966,1.016,4.68,19.73
6,Uber,7_15+ min,190685,1.25,11.93,21.85,37.99,3.796,1.057,5.48,21.40
7,Lyft,1_<2 min,489732,9.85,8.16,13.65,23.74,4.702,0.888,2.74,15.57
8,Lyft,2_2-<4 min,1912884,38.49,7.87,12.78,20.73,4.771,0.880,2.57,14.55
9,Lyft,3_4-<6 min,1438361,28.94,8.08,13.21,21.50,4.709,0.893,2.70,14.77


Median pay per mile (rows: company, distance band; columns: wait band)


wait_band              1_<2 min  2_2-<4 min  3_4-<6 min  4_6-<8 min  \
company distance_band                                                 
Lyft    1_<1 mi           7.702       7.662       7.726       7.878   
        2_1-<2 mi         5.516       5.436       5.426       5.505   
        3_2-<3 mi         4.973       4.899       4.891       4.938   
        4_3-<5 mi         4.357       4.338       4.333       4.376   
        5_5-<10 mi        3.525       3.530       3.526       3.545   
        6_10-<20 mi       3.063       2.978       2.970       2.985   
        7_20+ mi          2.668       2.646       2.637       2.643   
Uber    1_<1 mi           6.968       7.143       7.373       7.824   
        2_1-<2 mi         5.459       5.471       5.518       5.719   
        3_2-<3 mi         4.935       4.919       4.925       5.022   
        4_3-<5 mi         4.326       4.327       4.341       4.394   
        5_5-<10 mi        3.454       3.444       3.446       3.465   
        6_10-<20 mi       2.905       2.915       2.923       2.939   
        7_20+ mi          2.622       2.616       2.628       2.623   

wait_band              5_8-<10 min  6_10-<15 min  7_15+ min  
company distance_band                                        
Lyft    1_<1 mi              8.090         8.370      8.794  
        2_1-<2 mi            5.586         5.753      5.982  
        3_2-<3 mi            4.998         5.083      5.179  
        4_3-<5 mi            4.411         4.474      4.565  
        5_5-<10 mi           3.565         3.571      3.564  
        6_10-<20 mi          2.989         3.015      2.985  
        7_20+ mi             2.643         2.630      2.611  
Uber    1_<1 mi              8.283         8.596      9.257  
        2_1-<2 mi            5.911         6.027      6.217  
        3_2-<3 mi            5.107         5.079      5.083  
        4_3-<5 mi            4.403         4.339      4.295  
        5_5-<10 mi           3.448         3.393      3.437  
        6_10-<20 mi          2.932         2.913      3.012  
        7_20+ mi             2.609         2.608      2.732

Median pay per trip (rows: company, distance band; columns: wait band)


wait_band              1_<2 min  2_2-<4 min  3_4-<6 min  4_6-<8 min  \
company distance_band                                                 
Lyft    1_<1 mi            5.47        5.47        5.47        5.47   
        2_1-<2 mi          8.07        7.97        7.96        8.07   
        3_2-<3 mi         12.15       11.98       11.97       12.08   
        4_3-<5 mi         16.69       16.60       16.59       16.76   
        5_5-<10 mi        25.33       24.31       24.34       24.60   
        6_10-<20 mi       41.84       40.20       40.12       40.78   
        7_20+ mi          73.66       69.99       70.17       71.40   
Uber    1_<1 mi            4.92        5.01        5.17        5.52   
        2_1-<2 mi          7.94        8.00        8.07        8.38   
        3_2-<3 mi         12.05       12.04       12.08       12.34   
        4_3-<5 mi         16.55       16.56       16.65       16.93   
        5_5-<10 mi        24.19       24.25       24.51       24.96   
        6_10-<20 mi       39.34       39.62       39.97       40.88   
        7_20+ mi          70.70       69.32       69.21       70.68   

wait_band              5_8-<10 min  6_10-<15 min  7_15+ min  
company distance_band                                        
Lyft    1_<1 mi               5.47          5.47       5.83  
        2_1-<2 mi             8.20          8.40       8.66  
        3_2-<3 mi            12.22         12.43      12.66  
        4_3-<5 mi            16.86         17.08      17.54  
        5_5-<10 mi           24.88         25.18      25.73  
        6_10-<20 mi          41.91         43.17      43.48  
        7_20+ mi             72.28         72.88      72.11  
Uber    1_<1 mi               5.93          6.14       6.48  
        2_1-<2 mi             8.67          8.86       9.15  
        3_2-<3 mi            12.54         12.50      12.51  
        4_3-<5 mi            17.03         16.84      16.80  
        5_5-<10 mi           25.01         24.75      25.21  
        6_10-<20 mi          41.63         41.83      43.50  
        7_20+ mi             70.33         69.85      76.19

Cell sizes (rows: company, distance band; columns: wait band)


wait_band              1_<2 min  2_2-<4 min  3_4-<6 min  4_6-<8 min  \
company distance_band                                                 
Lyft    1_<1 mi         59930.0    245005.0    175063.0     75575.0   
        2_1-<2 mi      125262.0    506421.0    363308.0    156360.0   
        3_2-<3 mi       75545.0    322390.0    239448.0    103033.0   
        4_3-<5 mi       80808.0    356107.0    271517.0    115215.0   
        5_5-<10 mi      85961.0    320602.0    253867.0    108449.0   
        6_10-<20 mi     48939.0    135910.0    111498.0     52294.0   
        7_20+ mi        13249.0     26408.0     23623.0     13469.0   
Uber    1_<1 mi        227648.0    797321.0    444291.0    163383.0   
        2_1-<2 mi      516503.0   1735850.0    955147.0    360138.0   
        3_2-<3 mi      278793.0   1000785.0    578964.0    226517.0   
        4_3-<5 mi      293189.0   1108674.0    671193.0    273285.0   
        5_5-<10 mi     280194.0   1121909.0    739750.0    328270.0   
        6_10-<20 mi    132710.0    559547.0    409235.0    212872.0   
        7_20+ mi        25610.0    106332.0     84332.0     52332.0   

wait_band              5_8-<10 min  6_10-<15 min  7_15+ min  
company distance_band                                        
Lyft    1_<1 mi            31886.0       22337.0     4617.0  
        2_1-<2 mi          66062.0       45804.0     9102.0  
        3_2-<3 mi          43580.0       30250.0     6130.0  
        4_3-<5 mi          47542.0       32574.0     6742.0  
        5_5-<10 mi         44979.0       31864.0     7512.0  
        6_10-<20 mi        26139.0       22238.0     6415.0  
        7_20+ mi            8207.0        8365.0     2598.0  
Uber    1_<1 mi            62494.0       43169.0    10573.0  
        2_1-<2 mi         144899.0      104594.0    26605.0  
        3_2-<3 mi          94725.0       74902.0    20932.0  
        4_3-<5 mi         120374.0      102482.0    31065.0  
        5_5-<10 mi        157796.0      144100.0    47385.0  
        6_10-<20 mi       119294.0      119570.0    40490.0  
        7_20+ mi           34119.0       35475.0    13514.0

In [8]:
bor = run(p, "pay_by_borough_company"); display(bor)
zn = run(p, "pay_by_zone")
print(len(zn), "zones | rows:", int(zn["n"].sum()), "(expect 260 and 20,404,892)")
display(zn.groupby("sample_tier").agg(zones=("PULocationID", "count"), rows=("n", "sum")))
cols = ["pu_borough", "pu_zone", "is_airport_zone", "n", "pct_uber", "pay_p50", "ppm_p50", "ppmin_p50", "median_miles", "median_minutes", "ppm_p50_2_5mi"]
ad = zn[zn["sample_tier"] == "3_adequate"]
print("Highest median pay per mile (adequate zones)"); display(ad.sort_values("ppm_p50", ascending=False)[cols].head(15))
print("Lowest median pay per mile (adequate zones)"); display(ad.sort_values("ppm_p50")[cols].head(15))
display(run(p, "pay_airport_split"))
dp = run(p, "pay_by_daypart_company"); display(dp)
hr = run(p, "pay_by_hour_company")
display(hr.pivot(index="request_hour", columns="company", values=["pay_p50", "ppm_p50", "ppmin_p50"]))
zc = run(p, "pay_by_zone_company")

,company,pu_borough,n,pay_p50,ppm_p50,ppmin_p50,median_miles,median_minutes
0,All,Bronx,2591558,12.68,4.372,0.935,2.80,13.48
1,All,Brooklyn,5458874,12.77,4.874,0.870,2.54,14.55
2,All,Manhattan,7784756,14.79,4.964,0.922,2.71,15.82
3,All,Queens,4260487,15.23,4.000,0.976,3.78,15.75
4,All,Staten Island,309217,12.12,3.764,1.012,3.08,12.02
5,Uber,Bronx,2078377,12.92,4.320,0.942,2.88,13.60
6,Uber,Brooklyn,4068691,12.95,4.837,0.873,2.59,14.67
7,Uber,Manhattan,5709645,15.02,4.989,0.939,2.70,15.73
8,Uber,Queens,3257878,15.60,3.954,0.983,3.93,15.98
9,Uber,Staten Island,241335,12.53,3.715,1.023,3.21,12.18


260 zones | rows: 20404892 (expect 260 and 20,404,892)


,zones,rows
sample_tier,,
1_very_small,4,53
2_small,6,3348
3_adequate,250,20401491


Highest median pay per mile (adequate zones)


,pu_borough,pu_zone,is_airport_zone,n,pct_uber,pay_p50,ppm_p50,ppmin_p50,median_miles,median_minutes,ppm_p50_2_5mi
86,Brooklyn,South Williamsburg,False,43007,76.9,7.79,6.687,0.806,1.05,9.92,5.128
60,Brooklyn,Borough Park,False,120950,78.6,9.32,5.782,0.812,1.51,11.67,5.171
117,Manhattan,Upper East Side South,False,171366,77.5,14.11,5.621,0.908,2.15,15.08,5.292
106,Manhattan,Midtown Center,False,239287,78.1,16.36,5.588,0.939,2.54,17.27,5.516
134,Manhattan,Greenwich Village North,False,121049,73.8,13.21,5.586,0.871,2.10,14.57,5.247
140,Manhattan,SoHo,False,114722,74.9,14.24,5.565,0.895,2.40,14.90,5.262
118,Manhattan,Midtown North,False,164633,75.6,16.63,5.558,0.931,2.50,17.57,5.518
136,Manhattan,Meatpacking/West Village West,False,116266,73.0,13.77,5.536,0.892,2.18,14.77,5.298
167,Manhattan,Randalls Island,False,2350,69.1,36.72,5.514,1.811,5.71,19.72,NaN
114,Manhattan,West Village,False,185906,72.7,13.13,5.500,0.887,2.12,14.28,5.124


Lowest median pay per mile (adequate zones)


,pu_borough,pu_zone,is_airport_zone,n,pct_uber,pay_p50,ppm_p50,ppmin_p50,median_miles,median_minutes,ppm_p50_2_5mi
171,Queens,JFK Airport,True,366536,80.4,50.62,2.947,1.228,17.58,40.13,3.320
40,Bronx,City Island,False,2903,80.7,23.37,2.972,1.176,7.40,19.73,NaN
172,Queens,LaGuardia Airport,True,361220,75.0,31.45,3.153,1.148,9.52,27.13,3.743
254,Staten Island,Charleston/Tottenville,False,8789,83.0,14.00,3.225,1.177,4.25,12.03,3.409
227,Queens,Oakland Gardens,False,15301,78.6,16.29,3.267,1.084,4.93,14.57,3.524
257,Staten Island,Rossville/Woodrow,False,5416,83.0,12.64,3.325,1.155,3.61,11.38,3.451
39,Bronx,Pelham Bay Park,False,3430,83.9,10.22,3.336,1.088,2.99,9.43,3.244
256,Staten Island,Arden Heights,False,5841,79.6,12.23,3.405,1.095,3.50,11.27,3.489
224,Queens,Glen Oaks,False,19105,78.1,17.77,3.406,1.120,5.37,15.57,3.938
241,Staten Island,Bloomfield/Emerson Hill,False,31721,79.1,13.59,3.409,1.064,3.87,12.80,3.557


,company,zone_group,n,pay_p50,ppm_p50,ppmin_p50,median_miles,median_minutes
0,Uber,airport zones,565555,39.70,3.048,1.190,11.78,32.62
1,Uber,other zones,14790371,13.70,4.664,0.920,2.74,14.67
2,Lyft,airport zones,162201,38.72,3.062,1.171,11.69,32.95
3,Lyft,other zones,4886765,12.97,4.790,0.887,2.60,14.62


,company,daypart,day_type,n,pay_p50,ppm_p50,ppmin_p50,median_miles,median_minutes
0,All,1_overnight_0-5,Mon-Fri,1486411,16.27,3.905,1.093,3.81,14.47
1,All,1_overnight_0-5,Sat-Sun,943339,14.49,3.975,1.000,3.59,14.55
2,All,2_morning_6-9,Mon-Fri,2961932,14.09,4.888,0.926,2.68,15.08
3,All,2_morning_6-9,Sat-Sun,604513,13.51,3.849,0.995,3.47,13.80
4,All,3_midday_10-15,Mon-Fri,4040433,13.91,4.942,0.873,2.63,15.88
5,All,3_midday_10-15,Sat-Sun,1576569,13.27,4.628,0.896,2.74,14.82
6,All,4_evening_peak_16-19,Mon-Fri,3507074,13.32,5.041,0.871,2.47,15.15
7,All,4_evening_peak_16-19,Sat-Sun,1252030,13.88,4.838,0.887,2.71,15.63
8,All,5_late_evening_20-23,Mon-Fri,2901826,13.77,4.304,0.943,3.07,14.52
9,All,5_late_evening_20-23,Sat-Sun,1131539,14.37,4.467,0.954,3.07,14.93


pay_p50               ppm_p50               ppmin_p50         \
company          All   Lyft   Uber     All   Lyft   Uber       All   Lyft   
request_hour                                                                
0              14.54  13.33  15.06   4.169  4.240  4.139     1.003  0.954   
1              15.10  13.92  15.62   4.158  4.220  4.132     1.030  0.977   
2              15.31  14.47  15.68   4.068  4.129  4.042     1.050  1.002   
3              15.87  15.20  16.16   3.804  3.879  3.770     1.078  1.029   
4              17.35  16.91  17.55   3.486  3.567  3.447     1.128  1.092   
5              16.56  16.09  16.77   3.525  3.605  3.489     1.088  1.062   
6              15.73  14.38  16.26   4.064  4.068  4.062     1.041  0.969   
7              14.57  13.33  15.05   4.809  4.859  4.788     0.959  0.906   
8              13.43  12.88  13.63   4.946  5.098  4.891     0.908  0.877   
9              13.11  12.91  13.19   4.684  4.830  4.637     0.895  0.874   
10             13.30  13.16  13.35   4.648  4.778  4.605     0.896  0.877   
11             13.51  13.21  13.62   4.732  4.884  4.681     0.890  0.869   
12             13.65  13.29  13.78   4.777  4.926  4.728     0.888  0.868   
13             13.80  13.36  13.95   4.828  4.984  4.774     0.883  0.862   
14             13.95  13.48  14.11   4.995  5.145  4.947     0.868  0.848   
15             14.02  13.45  14.21   5.032  5.184  4.983     0.864  0.843   
16             13.84  13.25  14.03   5.076  5.209  5.037     0.863  0.841   
17             13.76  13.12  13.97   5.158  5.283  5.118     0.863  0.838   
18             13.32  12.59  13.56   4.973  5.133  4.921     0.879  0.852   
19             13.01  12.50  13.18   4.749  4.885  4.704     0.896  0.869   
20             13.44  12.82  13.65   4.519  4.643  4.477     0.921  0.892   
21             14.03  13.16  14.35   4.394  4.513  4.350     0.946  0.910   
22             14.23  13.35  14.56   4.290  4.435  4.235     0.954  0.920   
23             14.14  13.30  14.46   4.167  4.295  4.117     0.969  0.933   

                     
company        Uber  
request_hour         
0             1.024  
1             1.054  
2             1.071  
3             1.099  
4             1.145  
5             1.100  
6             1.072  
7             0.984  
8             0.920  
9             0.902  
10            0.902  
11            0.898  
12            0.896  
13            0.891  
14            0.875  
15            0.871  
16            0.870  
17            0.871  
18            0.889  
19            0.907  
20            0.932  
21            0.961  
22            0.968  
23            0.985

In [9]:
fa = run(p, "fare_component_audit"); display(fa.T)
ptf = run(p, "pay_to_fare_by_company"); display(ptf)
pgd = run(p, "pay_gt_fare_by_distance_band"); display(pgd)

,0,1
company,Uber,Lyft
n_rows,15356455,5049211
n_base_fare_positive,15353230,5035232
pct_rows_tolls,10.9,18.85
pct_rows_bcf,99.95,98.52
pct_rows_sales_tax,97.38,96.42
pct_rows_congestion_surcharge,36.68,40.46
pct_rows_airport_fee,7.58,7.51
pct_rows_cbd_fee,30.2,32.91
pct_rows_tips,17.95,19.94


,company,n,n_pay_gt_fare,pct_pay_gt_fare,mean_ratio,p01,p05,p25,p50,p75,p95,p99
0,Uber,15352917,2355102,15.340,0.7843,0.4495,0.4972,0.5971,0.7334,0.9098,1.2164,1.5669
1,Lyft,5035232,304766,6.053,0.7625,0.4458,0.5326,0.6618,0.7493,0.8443,1.0213,1.2335


,company,distance_band,n,pct_pay_gt_fare,pay_to_base_fare_p50
0,Uber,0_zero miles,2398,5.96,0.5701
1,Uber,1_<1 mi,1751887,3.71,0.5508
2,Uber,2_1-<2 mi,3851535,7.70,0.6632
3,Uber,3_2-<3 mi,2283477,12.04,0.7319
4,Uber,4_3-<5 mi,2614562,17.38,0.7857
5,Uber,5_5-<10 mi,2849266,25.16,0.8566
6,Uber,6_10-<20 mi,1637500,28.10,0.8832
7,Uber,7_20+ mi,362292,24.08,0.8414
8,Lyft,0_zero miles,208,8.65,0.7211
9,Lyft,1_<1 mi,617525,3.00,0.6794


In [15]:
r_pay = run(p, "robust_pay_level")     # 10a

In [16]:
r_ppm = run(p, "robust_ppm")           # 10b

In [17]:
r_ppmin = run(p, "robust_ppmin")       # 10c

In [18]:
rob = pd.concat([r_pay, r_ppm, r_ppmin], ignore_index=True)
s0 = rob[rob["scenario"].str.startswith("S0")].set_index(["metric", "company"])
def pct_vs_s0(row, col):
    base = s0.loc[(row["metric"], row["company"]), col]
    return None if pd.isna(base) or base == 0 else round(100 * (row[col] / base - 1), 3)
for col in ["n", "mean_value", "total_value", "p10", "p50", "p90", "p99"]:
    rob[f"{col}_pct_vs_S0"] = rob.apply(lambda r: pct_vs_s0(r, col), axis=1)
display(rob.sort_values(["metric", "company", "scenario"]))

,metric,scenario,company,n,mean_value,total_value,p10,p50,p90,p99,n_pct_vs_S0,mean_value_pct_vs_S0,total_value_pct_vs_S0,p10_pct_vs_S0,p50_pct_vs_S0,p90_pct_vs_S0,p99_pct_vs_S0
4,pay,S0 all rows,Lyft,5049211,17.7810,8.977982e+07,5.4700,13.3200,34.5600,74.0200,0.000,0.000,0.000,0.000,0.000,0.000,0.000
5,pay,S1 pay > 0,Lyft,5049211,17.7810,8.977982e+07,5.4700,13.3200,34.5600,74.0200,0.000,0.000,0.000,0.000,0.000,0.000,0.000
6,pay,S2 base fare > 0,Lyft,5035232,17.7793,8.952276e+07,5.4700,13.3200,34.5600,73.9900,-0.277,-0.010,-0.286,0.000,0.000,0.000,-0.041
7,pay,S3 no money flag,Lyft,5035024,17.7796,8.952081e+07,5.4700,13.3200,34.5600,73.9900,-0.281,-0.008,-0.289,0.000,0.000,0.000,-0.041
0,pay,S0 all rows,Uber,15356455,18.9370,2.908052e+08,5.6700,14.1600,37.6600,77.9800,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,pay,S1 pay > 0,Uber,15351917,18.9429,2.908101e+08,5.6700,14.1600,37.6700,77.9900,-0.030,0.031,0.002,0.000,0.000,0.027,0.013
2,pay,S2 base fare > 0,Uber,15353230,18.9385,2.907670e+08,5.6700,14.1600,37.6700,77.9900,-0.021,0.008,-0.013,0.000,0.000,0.027,0.013
3,pay,S3 no money flag,Uber,15346774,18.9458,2.907573e+08,5.6700,14.1700,37.6700,78.0000,-0.063,0.046,-0.016,0.000,0.071,0.027,0.026
12,pay_per_mile,S0 all eligible,Lyft,5049003,5.2686,NaN,2.9836,4.7305,7.7169,14.9853,0.000,0.000,NaN,0.000,0.000,0.000,0.000
13,pay_per_mile,S1 pay > 0,Lyft,5049003,5.2686,NaN,2.9836,4.7305,7.7169,14.9853,0.000,0.000,NaN,0.000,0.000,0.000,0.000


In [19]:
out = Path("dashboard/data"); out.mkdir(parents=True, exist_ok=True)
for name, df in [("2f_reconcile", rec), ("2f_pay_distribution", pay_dist), ("2f_pay_mean_sensitivity", pay_mean),
                 ("2f_ppm_distribution", ppm), ("2f_ppmin_distribution", ppmin), ("2f_ratio_denominator_diagnostics", diag),
                 ("2f_pay_by_distance_band", dist), ("2f_pay_by_duration_band", dur), ("2f_pay_by_wait_band", wb),
                 ("2f_pay_by_wait_within_distance", wwd), ("2f_pay_by_borough_company", bor), ("2f_pay_by_zone", zn),
                 ("2f_pay_by_zone_company", zc), ("2f_pay_by_daypart_company", dp), ("2f_pay_by_hour_company", hr),
                 ("2f_fare_component_audit", fa), ("2f_pay_to_fare_by_company", ptf), ("2f_pay_gt_fare_by_distance_band", pgd),
                 ("2f_money_filter_robustness", rob)]:
    df.to_csv(out / f"{name}.csv", index=False)